In [ ]:
import re
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()

try:
  from journalisticfeelings import export_preannotations, extract_articles_from_folder
except ImportError:
  !pip3 install 'git+ssh://git@git.lwp.rug.nl/team-data-science/2021/journalistic-feelings/journalisticfeelings.git'
  from journalisticfeelings import export_preannotations, extract_articles_from_folder


path = '/slow-data/journalisticfeelings/' #@param {"type": "string"}
os.chdir(path)

articles_path = './data/nexis-uni-export-unique.p3' #@param {"type": "string"}

predictions_path = './data/predictions/2023-02-13/' #@param {"type": "string"}
os.makedirs(predictions_path, exist_ok=True)

In [ ]:
#@markdown # Load articles

articles = pd.read_pickle(articles_path)

In [ ]:
#@markdown # Run models 

quote_model_name = 'resources/taggers/journalistic-quote-detection-xlm-roberta-base-set-350' #@param {"type": "string"}

pos_model_name = 'flair/upos-multi-fast' #@param {"type": "string"}

dependencies_model_name = "nl_core_news_sm" #@param {"type": "string"}

batch_size = 1024 #@param {"type": "integer"}

n_jobs = 2 #@param {"type": "integer"}

from multiprocessing import Pool, current_process

def init_models():
  gpu = current_process()._identity[0] % 2
  import os
  os.environ['CUDA_VISIBLE_DEVICES'] = str(gpu)

  import spacy
  from flair.models import SequenceTagger
  from spacy.pipeline.sentencizer import Sentencizer

  if 'quote_model' not in globals():
    globals()['quote_model'] = SequenceTagger.load(f'{quote_model_name}/final-model.pt')

  if 'pos_model' not in globals():
    globals()['pos_model'] = SequenceTagger.load(pos_model_name)

  if 'dependencies_model' not in globals():
    !python3 -m spacy download '{dependencies_model_name}'
    globals()['dependencies_model'] = spacy.load(dependencies_model_name)
    globals()['sentencizer_model'] = Sentencizer()


def quote_pos_and_dependency_detection(paragraph):
  from flair.data import Sentence
  from spacy.tokens.doc import Doc

  paragraph_sentence = Sentence(paragraph)
  pos_model.predict(paragraph_sentence)
  quote_model.predict(paragraph_sentence)
  paragraph_document = Doc(dependencies_model.vocab, [
      token.text for token in paragraph_sentence.tokens])
  sentencizer_model(paragraph_document)
  dependencies_model(paragraph_document)
  dependencies = pd.DataFrame({
    (sentence_idx, token.idx): {
        'token': token.text, 'label': token.dep_,
        'parent': token.head.idx, 'pos_spacy': token.pos_.lower()}
    for sentence_idx, sentence in enumerate(paragraph_document.sents)
    for token in sentence
  }).T

  dependencies['start'] = np.nan
  dependencies['end'] = np.nan
  start_idx = list(dependencies.columns).index('start')
  end_idx = list(dependencies.columns).index('end')
  for token in paragraph_sentence.tokens:
    assert dependencies.iloc[token.idx-1]['token'] == token.text
    dependencies.iloc[token.idx-1, start_idx] = token.start_pos
    dependencies.iloc[token.idx-1, end_idx] = token.end_pos

  dependencies['pos_flair'] = np.nan
  column_idx = list(dependencies.columns).index('pos_flair')
  for label in paragraph_sentence.get_labels('upos'):
    assert dependencies.iloc[label.data_point.idx-1]['token'] == label.data_point.text
    dependencies.iloc[label.data_point.idx-1, column_idx] = label.value
  
  dependencies['quote'] = False
  dependencies['quote idx'] = np.nan
  dependencies['pronoun'] = np.nan
  column_idx = list(dependencies.columns).index('quote')
  column_idx_idx = list(dependencies.columns).index('quote idx')
  column_idx_pronoun = list(dependencies.columns).index('pronoun')
  for quote_idx, label in enumerate(paragraph_sentence.get_labels('quote')):
    if label.value == "QUOTE":
      for token in label.data_point.tokens:
        assert dependencies.iloc[token.idx-1]['token'] == token.text
        dependencies.iloc[token.idx-1, column_idx] = True
        dependencies.iloc[token.idx-1, column_idx_idx] = quote_idx
    else:
      for token in label.data_point.tokens:
        dependencies.iloc[token.idx-1, column_idx_pronoun] = label.value

  
  dependencies = pd.concat([pd.DataFrame({(-1, -1): {'text': paragraph}}).T, dependencies])
  dependencies.index.names = ['sentence', 'token']
  return dependencies


texts = pd.Series(articles['body'].explode().unique())

first_batch = max((int(fn[6:-3]) for fn in os.listdir(predictions_path)), default=-1) + 1

done = {
  paragraph for fn in os.listdir(predictions_path)
  for paragraph in pd.read_pickle(f'{predictions_path}/{fn}')['text'].dropna()
}
texts_left = texts[~texts.isin(done)]
batches = texts_left.groupby((np.arange(len(texts_left)) // batch_size) + first_batch)

with Pool(n_jobs, initializer=init_models) as pool:
  for batch_num, batch in tqdm(batches, total=batches.ngroups):
    tqdm.pandas(leave=False)
    promises = batch.apply(lambda x: pool.apply_async(quote_pos_and_dependency_detection, (x,)))
    dependencies = promises.progress_apply(lambda x: x.get())
    dependencies = pd.concat(dependencies.tolist(), keys=np.arange(len(dependencies)), names=['paragraph', 'sentence', 'token'])
    dependencies.to_pickle(f'{predictions_path}/batch-{batch_num}.p3')
  pass